In [1]:
%pip install -U openai-whisper

Note: you may need to restart the kernel to use updated packages.


In [2]:
import whisper
model = whisper.load_model("base")

In [ ]:
import os

print(os.path.exists("data/1735404531.458927.wav"))


True


In [4]:
result = model.transcribe("data/1735404531.458927.wav")
print(result["text"])

c:\Users\shagu\anaconda3\envs\sentiment_pipeline\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 Hello. Hello. Hello, my name is Rick and I'm here. Hello, my name is Steven. I'm calling you from the Finance Department of the online company. You were speaking with Michael before and your manager is Mr. Omar, correct? Okay. All right. So, yeah, I was calling you to help you because we were trying to find a solution and the way to make very quick and easy withdrawal of your money on the platform back to your bank. So, I think we finally found an option and that's why I'm calling you to help you with that. So, it will just take another 5-10 minutes of the time. If you're available, I would like to guide you through the step and see if you can work out together. So, are you available for me to help you with that? Yeah. Okay. Okay. Wonderful. Now, I will ask you please open any desk and give me the end desk on your computer. Okay. And the numbers? One. And five. And five. Okay. I send a request. Okay. I will help you over here, okay? Okay. Okay. Okay. Okay. Okay. Okay. Okay. Now, the n

In [36]:
result = model.transcribe("data/1735404531.458927.wav", verbose = True)
print(result["text"])

Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:02.000]  Hello.
[00:02.000 --> 00:04.000]  Hello.
[00:04.000 --> 00:06.000]  Hello, my name is Rick and I'm here.
[00:06.000 --> 00:08.000]  Hello, my name is Steven.
[00:08.000 --> 00:14.000]  I'm calling you from the Finance Department of the online company.
[00:14.000 --> 00:19.000]  You were speaking with Michael before and your manager is Mr. Omar, correct?
[00:19.000 --> 00:20.000]  Okay.
[00:20.000 --> 00:25.000]  All right. So, yeah, I was calling you to help you because we were trying to find a solution
[00:25.000 --> 00:31.000]  and the way to make very quick and easy withdrawal of your money on the platform back to your bank.
[00:31.000 --> 00:36.000]  So, I think we finally found an option and that's why I'm calling you to help you with that.
[00:36.000 --> 00:39.000]  So, it will just take another 5-10 minutes of the time.
[00:39.000 --

In [ ]:
result['segments'] #

In [ ]:
%pip install pandas

In [38]:
import pandas as pd
speech = pd.DataFrame.from_dict(result['segments'])
speech.head()

,id,seek,start,end,text,tokens,temperature,avg_logprob,compression_ratio,no_speech_prob
0,0,0,0.0,2.0,Hello.,"[50364, 2425, 13, 50464]",0.0,-0.41821,1.511848,0.456676
1,1,0,2.0,4.0,Hello.,"[50464, 2425, 13, 50564]",0.0,-0.41821,1.511848,0.456676
2,2,0,4.0,6.0,"Hello, my name is Rick and I'm here.","[50564, 2425, 11, 452, 1315, 307, 11224, 293, ...",0.0,-0.41821,1.511848,0.456676
3,3,0,6.0,8.0,"Hello, my name is Steven.","[50664, 2425, 11, 452, 1315, 307, 12754, 13, 5...",0.0,-0.41821,1.511848,0.456676
4,4,0,8.0,14.0,I'm calling you from the Finance Department o...,"[50764, 286, 478, 5141, 291, 490, 264, 25765, ...",0.0,-0.41821,1.511848,0.456676


In [40]:
audio = whisper.load_audio("data/1735404531.458927.wav")
audio = whisper.pad_or_trim(audio)

In [41]:
mel = whisper.log_mel_spectrogram(audio).to(model.device)

_, probs = model.detect_language(mel)
probs

{'my': 0.00015904137399047613,
 'si': 0.00018021765572484583,
 'bg': 9.277429489884526e-05,
 'ka': 4.357962097856216e-05,
 'de': 0.00746602937579155,
 'pa': 2.1590003598248586e-05,
 'he': 0.006143268663436174,
 'hi': 0.0006869824137538671,
 'sl': 0.0007053794688545167,
 'ba': 3.9640035254251416e-08,
 'as': 1.2018916095257737e-05,
 'ta': 0.00022855897259432822,
 'pl': 0.000644203566480428,
 'ar': 0.009716851636767387,
 'uk': 0.00015124643687158823,
 'zh': 0.0012639878550544381,
 'bs': 3.7062483897898346e-05,
 'ht': 0.00011197698040632531,
 'mk': 2.6868630811804906e-05,
 'so': 5.332812634151196e-06,
 'ru': 0.0011453022016212344,
 'cy': 0.0019248421303927898,
 'az': 2.0441279048100114e-05,
 'fa': 0.0007586955325677991,
 'hy': 0.00011068928870372474,
 'bo': 9.112020052270964e-05,
 'bn': 0.0007096979534253478,
 'tt': 1.2230858601469663e-06,
 'ca': 0.00012112171680200845,
 'oc': 0.00011727948003681377,
 'la': 0.002623901469632983,
 'mn': 5.597517156274989e-05,
 'vi': 0.0007217009551823139,
 

In [42]:
print(f"Detected language: {max(probs, key=probs.get)}")

Detected language: en


In [43]:
import pandas as pd
from transformers import pipeline

# 1. Initialize the Sentiment Analysis Pipeline
# We use the TabularisAI model specifically for its multilingual/call-center capabilities
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="tabularisai/multilingual-sentiment-analysis"
)

# 2. Extract segments from your Whisper 'result'
segments = result['segments']

# 3. Process segments to create a list of results
processed_data = []

print("--- Running Sentiment Analysis ---")
for seg in segments:
    # Get sentiment for the text segment
    sentiment_result = sentiment_analyzer(seg['text'])[0]
    
    processed_data.append({
        "start": seg['start'],
        "end": seg['end'],
        "text": seg['text'],
        "sentiment": sentiment_result['label'],
        "confidence": round(sentiment_result['score'], 4)
    })

# 4. Create the final DataFrame
df_sentiment = pd.DataFrame(processed_data)

# 5. Display the result
print("\nFinal Call Analysis:")
print(df_sentiment[['start', 'sentiment', 'text']])

Device set to use cpu


--- Running Sentiment Analysis ---

Final Call Analysis:
      start sentiment                                               text
0      0.00   Neutral                                             Hello.
1      2.00   Neutral                                             Hello.
2      4.00  Positive               Hello, my name is Rick and I'm here.
3      6.00  Positive                          Hello, my name is Steven.
4      8.00   Neutral   I'm calling you from the Finance Department o...
..      ...       ...                                                ...
135  673.84  Negative                                      I don't know.
136  676.84  Positive           Okay. All right. No problem. No problem.
137  679.84  Positive   Let's look like this. Let me click it and I'l...
138  682.84   Neutral   Let me double check it on my side. All right....
139  686.84  Positive                                           Bye now.

[140 rows x 3 columns]
